# مختبر اليوم الأول — كشف التسريب وصياغة مشكلة مغادرة العملاء

> هذا الدفتر مبني مباشرة من مواصفات مختبرات منافذ المعتمدة للدورة.


In [ ]:
from pathlib import Path

# يعمل محليًا داخل المستودع أو عند وضع الحزمة في مجلد مستقل
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data/raw"), Path("../../data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), DATA_CANDIDATES[0])
CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"
print("DATA_DIR:", DATA_DIR.resolve())


## هدف المختبر

يستكشف الطالب جدول العملاء الفعلي ويحوّل طلب العمل إلى مسألة تصنيف محددة: **هل سيتوقف العميل عن إجراء طلبات مكتملة خلال الثلاثين يومًا التالية لتاريخ اللقطة؟** كما يتعلم قاعدة الوقت التي تمنع إدخال معلومات من المستقبل إلى النموذج.

## السيناريو والبيانات

يحتوي `manafeth_customers.parquet` على **48,000 صف**؛ كل صف يمثل **عميلًا واحدًا عند تاريخ لقطة شهري**. الهدف هو `churned_30d`. القيمة `1` تعني أن العميل لم يجرِ طلبًا مكتملًا في الثلاثين يومًا التالية، والقيمة `0` تعني أنه استمر في الطلب.

| نوع العمود | أعمدة واقعية من الملف | قرار المختبر |
|---|---|---|
| هدف | `churned_30d` | يبقى في `y` فقط |
| معرّف | `customer_id` | يحتفظ به للتتبع والعرض، ويُستبعد من الخصائص |
| خصائص آمنة مبدئيًا | `city`، `city_tier`، `device`، `payment_method`، `tenure_months`، `orders_per_month`، `avg_basket_sar`، `days_since_last_order`، `distinct_categories`، `promo_usage_rate`، `avg_rating`، `last_promo_used` | مرشحة لتدخل `X` بعد الفحص |
| تسريب معلومات | `refund_issued`، `support_ticket_after_snapshot`، `next_month_orders` | تستبعد قطعًا لأنها تحدث بعد اللقطة أو تكشف المستقبل |
| تواريخ | `signup_date`، `snapshot_date` | لا تستخدم في أول مختبر؛ يناقش معناها وتوفرها فقط |

## خطوات التنفيذ بالتسلسل

1. افتح `lab01_manafeth_framing.ipynb` واقرأ جدول العملاء باستخدام `pd.read_parquet`.
2. اعرض أول خمسة صفوف، وأسماء الأعمدة، وأنواعها، وعدد القيم الناقصة.
3. اكتب في خلية نصية: «الصف يمثل …، والهدف هو …، والقرار الذي يساعده النموذج هو …».
4. صنّف الأعمدة في أربع مجموعات: هدف، خصائص آمنة، معرّفات، ومعلومات من المستقبل.
5. اختبر كل عمود مشكوك فيه بالسؤال: «هل كانت هذه القيمة موجودة عند تاريخ اللقطة قبل بدء الثلاثين يومًا التالية؟».
6. أنشئ قائمتي `safe_features` و`leak_columns`، ثم اطبعها أمام المدرب أو زميل المراجعة.

## ما يكتبه أو يشغله الطالب


In [ ]:
import pandas as pd

customers = pd.read_parquet(CUSTOMERS_PATH)
customers.head()
customers.info()
customers.isna().sum().sort_values(ascending=False)
customers["churned_30d"].value_counts(normalize=True)

safe_features = [
    "city", "city_tier", "device", "payment_method",
    "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating", "last_promo_used"
]

leak_columns = [
    "refund_issued",
    "support_ticket_after_snapshot",
    "next_month_orders"
]


## النتيجة المتوقعة

يظهر للطالب أن `avg_rating` و`last_promo_used` يحتويان على قيم ناقصة، وأن فئة المغادرة أقل من فئة الاستمرار. الأهم أن يسجل قرارًا واضحًا باستبعاد الأعمدة الثلاثة المسرّبة، لا أن يكتفي بقول إنها «أعمدة غير مناسبة».

## المهارات التي يراجعها الطالب

يصوغ الطالب مشكلة عمل، ويحدد وحدة التحليل والهدف والخصائص، ويقرأ جدول Parquet في Jupyter، ويفحص أنواع الأعمدة والنقص، ويطبق اختبار الوقت لاكتشاف تسريب البيانات.

## شرط التسليم قبل مغادرة اليوم

يسلم الطالب خلية نصية تحتوي صياغة المشكلة، وجدولًا أو قائمتين يوضحان الخصائص الآمنة والأعمدة المستبعدة مع سبب الاستبعاد. لا يقبل `next_month_orders` كخاصية في أي إجابة.

---
